In [ ]:
# ============================================================================
# CELL 1: Installation & Imports
# ============================================================================

!pip install ultralytics albumentations opencv-python-headless scikit-learn -q

import os
import sys
import random
import shutil
import yaml
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
from tqdm.auto import tqdm

import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

print("✅ All packages imported successfully!")

In [ ]:
# ============================================================================
# CELL 2: System Check & Random Seeds
# ============================================================================

def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# Check GPU
print("="*80)
print("SYSTEM INFORMATION")
print("="*80)
print(f"🖥️  GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"🐍 Python: {sys.version.split()[0]}")
print(f"🔥 PyTorch: {torch.__version__}")
print("="*80)

In [ ]:
# ============================================================================
# CELL 3: Configuration
# ============================================================================

@dataclass
class Config:
    """Configuration for the training pipeline."""

    # Paths
    raw_data_path: str = "/content/raw_dataset"
    output_path: str = "/content/processed_dataset"
    runs_path: str = "/content/runs"

    # Dataset splits
    train_ratio: float = 0.75
    val_ratio: float = 0.15
    test_ratio: float = 0.10

    # Image settings
    target_size: int = 256

    # Augmentation settings
    augmentation_multiplier: int = 6
    min_samples_per_class: int = 200

    # Color-specific settings
    hue_shift_limit: int = 3
    sat_shift_limit: int = 50
    val_shift_limit: int = 30
    brightness_limit: float = 0.30
    contrast_limit: float = 0.25
    gamma_limit: Tuple[int, int] = (75, 125)

    # Training settings
    model_name: str = 'yolo11s-cls.pt'
    epochs: int = 150
    batch_size: int = 64
    patience: int = 25
    image_size: int = 256

    # Optimization
    optimizer: str = 'AdamW'
    lr0: float = 0.0008
    lrf: float = 0.01
    weight_decay: float = 0.0005
    label_smoothing: float = 0.1

    # Experiment name
    experiment_name: str = "MultiColor_Bread_v1"

    # Random seed
    seed: int = 42

print("✅ Configuration class defined!")

In [ ]:
# ============================================================================
# CELL 4: Augmentation Pipeline Class
# ============================================================================

class ColorAugmentationPipeline:
    """Advanced augmentation pipeline optimized for multi-color classification."""

    def __init__(self, config: Config, mode: str = 'train'):
        self.config = config
        self.mode = mode
        self.target_size = config.target_size

        if mode == 'train':
            self.transform = self._create_train_transform()
        else:
            self.transform = self._create_val_transform()

    def _create_train_transform(self) -> A.Compose:
        """Create training augmentation pipeline."""
        return A.Compose([
            A.Resize(self.target_size, self.target_size),

            # Lighting normalization
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),

            # Color augmentation (CRITICAL for color classification)
            A.HueSaturationValue(
                hue_shift_limit=self.config.hue_shift_limit,
                sat_shift_limit=self.config.sat_shift_limit,
                val_shift_limit=self.config.val_shift_limit,
                p=0.95
            ),

            A.RandomBrightnessContrast(
                brightness_limit=self.config.brightness_limit,
                contrast_limit=self.config.contrast_limit,
                brightness_by_max=True,
                p=0.85
            ),

            A.RandomGamma(gamma_limit=self.config.gamma_limit, p=0.6),

            # Lighting simulation
            A.RandomShadow(
                shadow_roi=(0, 0.5, 1, 1),
                num_shadows_lower=1,
                num_shadows_upper=2,
                shadow_dimension=5,
                p=0.35
            ),

            A.RandomFog(
                fog_coef_lower=0.1,
                fog_coef_upper=0.3,
                alpha_coef=0.1,
                p=0.15
            ),

            # Geometric augmentation
            A.Rotate(limit=20, border_mode=cv2.BORDER_REFLECT, p=0.6),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1,
                scale_limit=0.25,
                rotate_limit=15,
                border_mode=cv2.BORDER_REFLECT,
                p=0.6
            ),
            A.Perspective(scale=(0.02, 0.05), p=0.3),

            # Quality degradation
            A.OneOf([
                A.MotionBlur(blur_limit=5, p=1.0),
                A.GaussianBlur(blur_limit=5, p=1.0),
                A.MedianBlur(blur_limit=5, p=1.0),
            ], p=0.4),

            A.OneOf([
                A.GaussNoise(var_limit=(5.0, 30.0), p=1.0),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
            ], p=0.35),

            A.ImageCompression(quality_lower=75, quality_upper=100, p=0.3),

            A.CoarseDropout(
                max_holes=3,
                max_height=int(self.target_size * 0.15),
                max_width=int(self.target_size * 0.15),
                min_holes=1,
                min_height=int(self.target_size * 0.05),
                min_width=int(self.target_size * 0.05),
                fill_value=0,
                p=0.2
            ),
        ])

    def _create_val_transform(self) -> A.Compose:
        """Create validation/test augmentation pipeline."""
        return A.Compose([
            A.Resize(self.target_size, self.target_size),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        ])

    def __call__(self, image: np.ndarray) -> np.ndarray:
        """Apply augmentation to image."""
        augmented = self.transform(image=image)
        return augmented['image']


def visualize_augmentations(image_path: str, config: Config, n_samples: int = 6):
    """Visualize augmentation effects on a sample image."""

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    augmentor = ColorAugmentationPipeline(config, mode='train')

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    axes[0].imshow(image)
    axes[0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0].axis('off')

    for i in range(1, min(n_samples + 1, 8)):
        aug_image = augmentor(image)
        axes[i].imshow(aug_image)
        axes[i].set_title(f'Augmented {i}', fontsize=12)
        axes[i].axis('off')

    for i in range(n_samples + 1, 8):
        axes[i].axis('off')

    plt.suptitle('Augmentation Preview', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.savefig('/content/augmentation_preview.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"✅ Augmentation preview saved to /content/augmentation_preview.png")

print("✅ Augmentation pipeline defined!")

In [ ]:
# ============================================================================
# CELL 5: Dataset Preparation Class
# ============================================================================

class DatasetPreparator:
    """Prepare dataset with augmentation and train/val/test splits."""

    def __init__(self, config: Config):
        self.config = config
        self.train_augmentor = ColorAugmentationPipeline(config, mode='train')
        self.val_augmentor = ColorAugmentationPipeline(config, mode='val')

    def prepare_dataset(
        self,
        input_path: str,
        output_path: str,
        balance_classes: bool = True
    ) -> Dict[str, int]:
        """Prepare complete dataset with augmentation and splits."""
        print("\n" + "="*80)
        print("DATASET PREPARATION")
        print("="*80)

        input_path = Path(input_path)
        output_path = Path(output_path)

        print("\n📊 Step 1: Scanning input dataset...")
        class_data = self._scan_dataset(input_path)

        print("\n✂️  Step 2: Creating train/val/test splits...")
        splits = self._create_splits(class_data)

        print("\n🔄 Step 3: Planning augmentation strategy...")
        aug_strategy = self._plan_augmentation(splits, balance_classes)

        print("\n🎨 Step 4: Generating augmented dataset...")
        stats = self._generate_dataset(splits, aug_strategy, output_path)

        print("\n📝 Step 5: Creating dataset summary...")
        self._create_summary(output_path, stats)

        print("\n" + "="*80)
        print("✅ DATASET PREPARATION COMPLETE")
        print("="*80)
        print(f"📁 Output: {output_path}")
        print(f"📊 Total images: {sum(stats['total'].values())}")
        print(f"🎓 Classes: {len(stats['classes'])}")

        return stats

    def _scan_dataset(self, input_path: Path) -> Dict[str, List[Path]]:
        """Scan input dataset and organize by class."""
        class_data = defaultdict(list)

        for class_dir in input_path.iterdir():
            if not class_dir.is_dir():
                continue

            class_name = class_dir.name
            image_files = list(class_dir.glob('*.jpg')) + \
                         list(class_dir.glob('*.jpeg')) + \
                         list(class_dir.glob('*.png'))

            class_data[class_name] = image_files
            print(f"  📦 {class_name}: {len(image_files)} images")

        if not class_data:
            raise ValueError(f"No class folders found in {input_path}")

        print(f"\n  ✅ Found {len(class_data)} classes, "
              f"{sum(len(v) for v in class_data.values())} total images")

        return dict(class_data)

    def _create_splits(
        self,
        class_data: Dict[str, List[Path]]
    ) -> Dict[str, Dict[str, List[Path]]]:
        """Split data into train/val/test sets."""
        splits = {'train': {}, 'val': {}, 'test': {}}

        for class_name, image_paths in class_data.items():
            train_paths, temp_paths = train_test_split(
                image_paths,
                test_size=(self.config.val_ratio + self.config.test_ratio),
                random_state=self.config.seed
            )

            val_ratio_adjusted = self.config.val_ratio / (self.config.val_ratio + self.config.test_ratio)
            val_paths, test_paths = train_test_split(
                temp_paths,
                test_size=(1 - val_ratio_adjusted),
                random_state=self.config.seed
            )

            splits['train'][class_name] = train_paths
            splits['val'][class_name] = val_paths
            splits['test'][class_name] = test_paths

            print(f"  {class_name}: "
                  f"train={len(train_paths)}, "
                  f"val={len(val_paths)}, "
                  f"test={len(test_paths)}")

        return splits

    def _plan_augmentation(
        self,
        splits: Dict[str, Dict[str, List[Path]]],
        balance_classes: bool
    ) -> Dict[str, Dict[str, int]]:
        """Plan how many augmented samples to create per class."""
        aug_strategy = {'train': {}, 'val': {}, 'test': {}}

        train_counts = {cls: len(paths) for cls, paths in splits['train'].items()}

        if balance_classes:
            max_count = max(train_counts.values())
            target_count = max(
                max_count * self.config.augmentation_multiplier,
                self.config.min_samples_per_class
            )
        else:
            target_count = None

        for split_name in ['train', 'val', 'test']:
            for class_name, paths in splits[split_name].items():
                current_count = len(paths)

                if split_name == 'train':
                    if balance_classes:
                        desired_count = target_count
                    else:
                        desired_count = current_count * self.config.augmentation_multiplier

                    augs_per_image = max(1, desired_count // current_count)
                else:
                    augs_per_image = 1

                aug_strategy[split_name][class_name] = augs_per_image

        print("\n  Augmentation strategy:")
        for class_name in splits['train'].keys():
            train_orig = len(splits['train'][class_name])
            train_augs = aug_strategy['train'][class_name]
            train_final = train_orig * train_augs

            print(f"    {class_name}: "
                  f"{train_orig} → {train_final} images "
                  f"({train_augs}x augmentation)")

        return aug_strategy

    def _generate_dataset(
        self,
        splits: Dict[str, Dict[str, List[Path]]],
        aug_strategy: Dict[str, Dict[str, int]],
        output_path: Path
    ) -> Dict:
        """Generate augmented dataset and save to disk."""
        stats = {
            'classes': [],
            'train': {},
            'val': {},
            'test': {},
            'total': {}
        }

        for split_name in ['train', 'val', 'test']:
            print(f"\n  Processing {split_name} set...")

            split_data = splits[split_name]
            split_output = output_path / split_name

            for class_name, image_paths in tqdm(split_data.items(), desc=f"  {split_name}"):
                class_output = split_output / class_name
                class_output.mkdir(parents=True, exist_ok=True)

                if split_name == 'train':
                    augmentor = self.train_augmentor
                else:
                    augmentor = self.val_augmentor

                augs_per_image = aug_strategy[split_name][class_name]

                saved_count = 0
                for img_idx, img_path in enumerate(image_paths):
                    image = cv2.imread(str(img_path))
                    if image is None:
                        print(f"    ⚠️  Failed to load: {img_path}")
                        continue

                    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                    for aug_idx in range(augs_per_image):
                        aug_image = augmentor(image)
                        aug_image_bgr = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)

                        if augs_per_image == 1:
                            filename = f"{img_path.stem}.jpg"
                        else:
                            filename = f"{img_path.stem}_aug{aug_idx:03d}.jpg"

                        output_file = class_output / filename
                        cv2.imwrite(str(output_file), aug_image_bgr,
                                   [cv2.IMWRITE_JPEG_QUALITY, 95])
                        saved_count += 1

                stats[split_name][class_name] = saved_count

                if class_name not in stats['classes']:
                    stats['classes'].append(class_name)

        for class_name in stats['classes']:
            stats['total'][class_name] = (
                stats['train'].get(class_name, 0) +
                stats['val'].get(class_name, 0) +
                stats['test'].get(class_name, 0)
            )

        return stats

    def _create_summary(self, output_path: Path, stats: Dict):
        """Create dataset summary file."""
        summary_path = output_path / 'dataset_summary.txt'

        with open(summary_path, 'w') as f:
            f.write("="*80 + "\n")
            f.write("DATASET SUMMARY\n")
            f.write("="*80 + "\n\n")

            f.write(f"Classes: {len(stats['classes'])}\n")
            f.write(f"Class names: {', '.join(sorted(stats['classes']))}\n\n")

            f.write("Distribution:\n")
            f.write("-" * 80 + "\n")
            f.write(f"{'Class':<25} {'Train':>10} {'Val':>10} {'Test':>10} {'Total':>10}\n")
            f.write("-" * 80 + "\n")

            for class_name in sorted(stats['classes']):
                f.write(f"{class_name:<25} "
                       f"{stats['train'][class_name]:>10} "
                       f"{stats['val'][class_name]:>10} "
                       f"{stats['test'][class_name]:>10} "
                       f"{stats['total'][class_name]:>10}\n")

            f.write("-" * 80 + "\n")
            f.write(f"{'TOTAL':<25} "
                   f"{sum(stats['train'].values()):>10} "
                   f"{sum(stats['val'].values()):>10} "
                   f"{sum(stats['test'].values()):>10} "
                   f"{sum(stats['total'].values()):>10}\n")
            f.write("="*80 + "\n")

        print(f"  ✅ Summary saved: {summary_path}")

print("✅ Dataset preparation class defined!")

In [ ]:
# ============================================================================
# CELL 6: Training Class
# ============================================================================

class YOLOTrainer:
    """YOLO model trainer with advanced configuration."""

    def __init__(self, config: Config):
        self.config = config
        self.model = None
        self.results = None

    def train(self, dataset_path: str) -> YOLO:
        """Train YOLO classification model."""
        print("\n" + "="*80)
        print("TRAINING YOLO CLASSIFICATION MODEL")
        print("="*80)

        print(f"\n📦 Loading model: {self.config.model_name}")
        self.model = YOLO(self.config.model_name)

        batch_size = self._get_optimal_batch_size()
        print(f"🎯 Batch size: {batch_size}")

        train_params = {
            'data': dataset_path,
            'epochs': self.config.epochs,
            'imgsz': self.config.image_size,
            'batch': batch_size,
            'patience': self.config.patience,
            'device': 0 if torch.cuda.is_available() else 'cpu',
            'optimizer': self.config.optimizer,
            'lr0': self.config.lr0,
            'lrf': self.config.lrf,
            'weight_decay': self.config.weight_decay,
            'warmup_epochs': 5,
            'warmup_momentum': 0.8,
            'warmup_bias_lr': 0.1,

            # Minimal augmentation (dataset is pre-augmented)
            'hsv_h': 0.005,
            'hsv_s': 0.2,
            'hsv_v': 0.2,
            'degrees': 5,
            'translate': 0.05,
            'scale': 0.15,
            'fliplr': 0.3,

            'label_smoothing': self.config.label_smoothing,
            'dropout': 0.0,

            'save': True,
            'save_period': 15,
            'project': self.config.runs_path,
            'name': self.config.experiment_name,
            'exist_ok': False,
            'pretrained': True,
            'verbose': True,
            'seed': self.config.seed,
            'deterministic': True,

            'workers': 2,
            'plots': True,
            'val': True,
        }

        print("\n🚀 Starting training...")
        print(f"⏱️  Estimated time: ~{self._estimate_training_time(batch_size)} minutes")
        print("="*80 + "\n")

        self.results = self.model.train(**train_params)

        print("\n" + "="*80)
        print("✅ TRAINING COMPLETE")
        print("="*80)
        print(f"📁 Results: {self.results.save_dir}")
        print(f"🏆 Best model: {self.results.save_dir}/weights/best.pt")

        return self.model

    def _get_optimal_batch_size(self) -> int:
        """Determine optimal batch size based on GPU memory."""
        if not torch.cuda.is_available():
            return 16

        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3

        if self.config.image_size <= 256:
            if gpu_memory >= 24:
                return 256
            elif gpu_memory >= 16:
                return 128
            elif gpu_memory >= 12:
                return 64
            else:
                return 32
        elif self.config.image_size <= 320:
            if gpu_memory >= 24:
                return 128
            elif gpu_memory >= 16:
                return 96
            elif gpu_memory >= 12:
                return 48
            else:
                return 24
        else:
            if gpu_memory >= 24:
                return 64
            elif gpu_memory >= 16:
                return 48
            else:
                return 24

    def _estimate_training_time(self, batch_size: int) -> int:
        """Estimate training time in minutes."""
        images_per_second = batch_size * 2
        total_images = self.config.epochs * 1000
        seconds = total_images / images_per_second
        return int(seconds / 60)

print("✅ Training class defined!")

In [ ]:
# ============================================================================
# CELL 7: Evaluation Class (FIXED)
# ============================================================================

class ModelEvaluator:
    """Comprehensive model evaluation and visualization."""

    def __init__(self, model_path: str, dataset_path: str):
        self.model = YOLO(model_path)
        self.dataset_path = Path(dataset_path)
        self.class_names = None
        self.y_true = []
        self.y_pred = []
        self.y_conf = []
        self.y_probs = []

    def evaluate(self, split: str = 'test') -> Dict:
        """Evaluate model on specified split."""
        print("\n" + "="*80)
        print(f"MODEL EVALUATION ({split.upper()} SET)")
        print("="*80)

        print("\n🔮 Generating predictions...")
        self._get_predictions(split)

        print("\n📊 Calculating metrics...")
        metrics = self._calculate_metrics()

        print("\n📈 Creating visualizations...")
        self._create_visualizations(split)

        print("\n🔍 Analyzing confusions...")
        self._analyze_confusions()

        self._print_summary(metrics)

        return metrics

    def _get_predictions(self, split: str):
        """Generate predictions for all images in split."""
        split_path = self.dataset_path / split

        if not split_path.exists():
            raise ValueError(f"Split not found: {split_path}")

        self.class_names = sorted([d.name for d in split_path.iterdir() if d.is_dir()])

        self.y_true = []
        self.y_pred = []
        self.y_conf = []
        self.y_probs = []

        for class_idx, class_name in enumerate(tqdm(self.class_names, desc="  Classes")):
            class_dir = split_path / class_name

            image_files = list(class_dir.glob('*.jpg')) + \
                         list(class_dir.glob('*.jpeg')) + \
                         list(class_dir.glob('*.png'))

            for img_path in image_files:
                try:
                    result = self.model(str(img_path), verbose=False)[0]

                    pred_idx = result.probs.top1
                    confidence = result.probs.top1conf.item()
                    probs = result.probs.data.cpu().numpy()

                    self.y_true.append(class_idx)
                    self.y_pred.append(pred_idx)
                    self.y_conf.append(confidence)
                    self.y_probs.append(probs)

                except Exception as e:
                    print(f"    ⚠️  Error on {img_path}: {e}")

        print(f"  ✅ Generated {len(self.y_pred)} predictions")

    def _calculate_metrics(self) -> Dict:
        """Calculate evaluation metrics."""
        correct = sum(1 for t, p in zip(self.y_true, self.y_pred) if t == p)
        accuracy = correct / len(self.y_true) if self.y_true else 0

        precision, recall, f1, support = precision_recall_fscore_support(
            self.y_true, self.y_pred, average=None, zero_division=0
        )

        conf_mean = np.mean(self.y_conf)
        conf_median = np.median(self.y_conf)
        conf_min = np.min(self.y_conf)
        conf_max = np.max(self.y_conf)

        low_conf_count = sum(1 for c in self.y_conf if c < 0.7)
        low_conf_ratio = low_conf_count / len(self.y_conf) if self.y_conf else 0

        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': support,
            'confidence': {
                'mean': conf_mean,
                'median': conf_median,
                'min': conf_min,
                'max': conf_max,
                'low_conf_count': low_conf_count,
                'low_conf_ratio': low_conf_ratio
            }
        }

        return metrics

    def _create_visualizations(self, split: str):
        """Create evaluation visualizations."""
        self._plot_confusion_matrix(split)
        self._plot_per_class_metrics(split)
        self._plot_confidence_distribution(split)
        self._plot_confidence_vs_accuracy(split)

    def _plot_confusion_matrix(self, split: str):
        """Plot confusion matrix."""
        cm = confusion_matrix(self.y_true, self.y_pred)

        plt.figure(figsize=(12, 10))

        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

        annotations = np.empty_like(cm, dtype=object)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                count = cm[i, j]
                percentage = cm_normalized[i, j] * 100
                annotations[i, j] = f'{count}\n({percentage:.1f}%)'

        sns.heatmap(
            cm_normalized,
            annot=annotations,
            fmt='',
            cmap='Blues',
            xticklabels=self.class_names,
            yticklabels=self.class_names,
            cbar_kws={'label': 'Normalized Frequency'},
            square=True,
            linewidths=0.5,
            linecolor='gray'
        )

        plt.ylabel('True Label', fontsize=13, fontweight='bold')
        plt.xlabel('Predicted Label', fontsize=13, fontweight='bold')
        plt.title(f'Confusion Matrix - {split.upper()} Set', fontsize=15, fontweight='bold', pad=20)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()

        output_path = f'/content/confusion_matrix_{split}.png'
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.show()

        print(f"  ✅ Confusion matrix saved: {output_path}")

    def _plot_per_class_metrics(self, split: str):
        """Plot per-class precision, recall, and F1 score."""
        precision, recall, f1, support = precision_recall_fscore_support(
            self.y_true, self.y_pred, average=None, zero_division=0
        )

        x = np.arange(len(self.class_names))
        width = 0.25

        fig, ax = plt.subplots(figsize=(14, 7))

        bars1 = ax.bar(x - width, precision, width, label='Precision', color='#3498db', alpha=0.8)
        bars2 = ax.bar(x, recall, width, label='Recall', color='#2ecc71', alpha=0.8)
        bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#e74c3c', alpha=0.8)

        def autolabel(bars):
            for bar in bars:
                height = bar.get_height()
                ax.annotate(f'{height:.3f}',
                           xy=(bar.get_x() + bar.get_width() / 2, height),
                           xytext=(0, 3),
                           textcoords="offset points",
                           ha='center', va='bottom',
                           fontsize=9, fontweight='bold')

        autolabel(bars1)
        autolabel(bars2)
        autolabel(bars3)

        ax.set_xlabel('Class', fontsize=13, fontweight='bold')
        ax.set_ylabel('Score', fontsize=13, fontweight='bold')
        ax.set_title(f'Per-Class Metrics - {split.upper()} Set', fontsize=15, fontweight='bold', pad=20)
        ax.set_xticks(x)
        ax.set_xticklabels(self.class_names, rotation=45, ha='right')
        ax.legend(loc='upper right', fontsize=11)
        ax.set_ylim([0, 1.05])
        ax.grid(axis='y', alpha=0.3, linestyle='--')

        plt.tight_layout()
        output_path = f'/content/per_class_metrics_{split}.png'
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.show()

        print(f"  ✅ Per-class metrics saved: {output_path}")

    def _plot_confidence_distribution(self, split: str):
        """Plot confidence score distribution."""
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        axes[0].hist(self.y_conf, bins=50, color='#3498db', alpha=0.7, edgecolor='black')
        axes[0].axvline(np.mean(self.y_conf), color='red', linestyle='--',
                       linewidth=2, label=f'Mean: {np.mean(self.y_conf):.3f}')
        axes[0].axvline(np.median(self.y_conf), color='green', linestyle='--',
                       linewidth=2, label=f'Median: {np.median(self.y_conf):.3f}')
        axes[0].set_xlabel('Confidence Score', fontsize=12, fontweight='bold')
        axes[0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
        axes[0].set_title('Overall Confidence Distribution', fontsize=13, fontweight='bold')
        axes[0].legend(fontsize=10)
        axes[0].grid(alpha=0.3, linestyle='--')

        correct_conf = [c for t, p, c in zip(self.y_true, self.y_pred, self.y_conf) if t == p]
        incorrect_conf = [c for t, p, c in zip(self.y_true, self.y_pred, self.y_conf) if t != p]

        axes[1].hist([correct_conf, incorrect_conf], bins=30,
                    color=['#2ecc71', '#e74c3c'], alpha=0.7,
                    label=['Correct', 'Incorrect'], edgecolor='black')
        axes[1].set_xlabel('Confidence Score', fontsize=12, fontweight='bold')
        axes[1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
        axes[1].set_title('Confidence by Prediction Correctness', fontsize=13, fontweight='bold')
        axes[1].legend(fontsize=10)
        axes[1].grid(alpha=0.3, linestyle='--')

        plt.tight_layout()
        output_path = f'/content/confidence_distribution_{split}.png'
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.show()

        print(f"  ✅ Confidence distribution saved: {output_path}")

    def _plot_confidence_vs_accuracy(self, split: str):
        """Plot accuracy at different confidence thresholds."""
        thresholds = np.arange(0.0, 1.01, 0.05)
        accuracies = []
        sample_counts = []

        for threshold in thresholds:
            filtered_indices = [i for i, c in enumerate(self.y_conf) if c >= threshold]

            if filtered_indices:
                filtered_true = [self.y_true[i] for i in filtered_indices]
                filtered_pred = [self.y_pred[i] for i in filtered_indices]
                accuracy = sum(1 for t, p in zip(filtered_true, filtered_pred) if t == p) / len(filtered_true)
                accuracies.append(accuracy)
                sample_counts.append(len(filtered_indices))
            else:
                accuracies.append(0)
                sample_counts.append(0)

        fig, ax1 = plt.subplots(figsize=(14, 7))

        color1 = '#3498db'
        ax1.set_xlabel('Confidence Threshold', fontsize=13, fontweight='bold')
        ax1.set_ylabel('Accuracy', fontsize=13, fontweight='bold', color=color1)
        line1 = ax1.plot(thresholds, accuracies, color=color1, linewidth=3,
                        marker='o', markersize=5, label='Accuracy')
        ax1.tick_params(axis='y', labelcolor=color1)
        ax1.grid(alpha=0.3, linestyle='--')
        ax1.set_ylim([0, 1.05])

        ax2 = ax1.twinx()
        color2 = '#e74c3c'
        ax2.set_ylabel('Sample Count', fontsize=13, fontweight='bold', color=color2)
        line2 = ax2.plot(thresholds, sample_counts, color=color2, linewidth=3,
                        marker='s', markersize=5, linestyle='--', label='Sample Count')
        ax2.tick_params(axis='y', labelcolor=color2)

        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='center right', fontsize=11)

        plt.title(f'Accuracy vs Confidence Threshold - {split.upper()} Set',
                 fontsize=15, fontweight='bold', pad=20)
        plt.tight_layout()

        output_path = f'/content/confidence_vs_accuracy_{split}.png'
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.show()

        print(f"  ✅ Confidence vs accuracy saved: {output_path}")

    def _analyze_confusions(self):
        """Analyze most confused class pairs."""
        cm = confusion_matrix(self.y_true, self.y_pred)

        confusion_pairs = []
        for i in range(len(self.class_names)):
            for j in range(len(self.class_names)):
                if i != j and cm[i, j] > 0:
                    confusion_pairs.append({
                        'true_class': self.class_names[i],
                        'pred_class': self.class_names[j],
                        'count': cm[i, j],
                        'percentage': cm[i, j] / cm[i].sum() * 100
                    })

        confusion_pairs.sort(key=lambda x: x['count'], reverse=True)

        print("\n  🔍 Top Confused Pairs:")
        print("  " + "-"*70)
        for pair in confusion_pairs[:10]:
            print(f"  {pair['true_class']:20s} → {pair['pred_class']:20s}: "
                  f"{pair['count']:3d} ({pair['percentage']:5.1f}%)")

        if not confusion_pairs:
            print("  ✨ No confusions found! Perfect classification!")

    def _print_summary(self, metrics: Dict):
        """Print evaluation summary."""
        print("\n" + "="*80)
        print("EVALUATION SUMMARY")
        print("="*80)

        print(f"\n📊 Overall Accuracy: {metrics['accuracy']*100:.2f}%")

        print(f"\n💯 Confidence Statistics:")
        print(f"  Mean:   {metrics['confidence']['mean']:.3f}")
        print(f"  Median: {metrics['confidence']['median']:.3f}")
        print(f"  Range:  [{metrics['confidence']['min']:.3f}, {metrics['confidence']['max']:.3f}]")
        print(f"  Low confidence (<0.7): {metrics['confidence']['low_conf_count']} "
              f"({metrics['confidence']['low_conf_ratio']*100:.1f}%)")

        print(f"\n📈 Per-Class Performance:")
        print("  " + "-"*70)
        print(f"  {'Class':<20} {'Precision':>10} {'Recall':>10} {'F1-Score':>10} {'Support':>10}")
        print("  " + "-"*70)

        for i, class_name in enumerate(self.class_names):
            print(f"  {class_name:<20} "
                  f"{metrics['precision'][i]:>10.3f} "
                  f"{metrics['recall'][i]:>10.3f} "
                  f"{metrics['f1'][i]:>10.3f} "
                  f"{int(metrics['support'][i]):>10d}")

        print("  " + "-"*70)
        print(f"  {'MACRO AVERAGE':<20} "
              f"{np.mean(metrics['precision']):>10.3f} "
              f"{np.mean(metrics['recall']):>10.3f} "
              f"{np.mean(metrics['f1']):>10.3f} "
              f"{int(np.sum(metrics['support'])):>10d}")
        print("="*80)

print("✅ Evaluation class defined!")

In [ ]:
# ============================================================================
# CELL 8: Configure Your Experiment
# ============================================================================

# OPTION 1: Use Google Drive (Recommended for persistence)
config = Config(
    raw_data_path="/content/drive/MyDrive/Colab Notebooks/ConvuyerBreadCounter/ClassificationDataset/InputDataset",
    output_path="/content/drive/MyDrive/Colab Notebooks/ConvuyerBreadCounter/ClassificationDataset/OutputDataset",
    runs_path="/content/drive/MyDrive/Colab Notebooks/ConvuyerBreadCounter/ClassificationDataset/Runs",

    target_size=256,
    augmentation_multiplier=6,

    model_name='yolo11s-cls.pt',
    epochs=100,
    batch_size=128,  # Will auto-adjust based on GPU
    patience=20,

    experiment_name="Balanced_v1"
)

# OPTION 2: Use local Colab storage (faster, but deleted when runtime ends)
# config = Config(
#     raw_data_path="/content/raw_dataset",
#     output_path="/content/processed_dataset",
#     runs_path="/content/runs",
#
#     target_size=256,
#     augmentation_multiplier=6,
#
#     model_name='yolo11s-cls.pt',
#     epochs=100,
#     batch_size=128,
#     patience=20,
#
#     experiment_name="Balanced_v1"
# )

# Create directories
Path(config.output_path).mkdir(parents=True, exist_ok=True)
Path(config.runs_path).mkdir(parents=True, exist_ok=True)

print(f"\n✅ Configuration set!")
print(f"📁 Input:  {config.raw_data_path}")
print(f"📁 Output: {config.output_path}")
print(f"📁 Runs:   {config.runs_path}")
print(f"🎯 Model:  {config.model_name}")
print(f"🔄 Augmentation: {config.augmentation_multiplier}x")

In [ ]:
# ============================================================================
# CELL 9: Preview Augmentations (Optional)
# ============================================================================

# Find a sample image to preview augmentations
sample_image = None
for class_dir in Path(config.raw_data_path).iterdir():
    if class_dir.is_dir():
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        if images:
            sample_image = str(images[0])
            break

if sample_image:
    print(f"📸 Sample image: {sample_image}")
    visualize_augmentations(sample_image, config, n_samples=7)
else:
    print("⚠️ No sample image found. Make sure your dataset is uploaded!")

In [ ]:
# ============================================================================
# CELL 10: Prepare Dataset
# ============================================================================

# Check if dataset already exists
if Path(config.output_path).exists() and any(Path(config.output_path).iterdir()):
    print(f"⚠️ Processed dataset already exists: {config.output_path}")
    print("   If you want to regenerate, delete the folder first.")
else:
    print("🔄 Preparing dataset...")
    preparator = DatasetPreparator(config)
    stats = preparator.prepare_dataset(
        input_path=config.raw_data_path,
        output_path=config.output_path,
        balance_classes=True
    )

    print("\n✅ Dataset preparation complete!")
    print(f"📊 Total images: {sum(stats['total'].values())}")

In [ ]:
# ============================================================================
# CELL 11: Train Model
# ============================================================================

trainer = YOLOTrainer(config)
model = trainer.train(config.output_path)

print("\n✅ Training complete!")
print(f"🏆 Best model: {config.runs_path}/{config.experiment_name}/weights/best.pt")

In [ ]:
# ============================================================================
# CELL 12: Evaluate on Validation Set
# ============================================================================

best_model_path = f"{config.runs_path}/{config.experiment_name}/weights/best.pt"

if Path(best_model_path).exists():
    print("📊 Evaluating on VALIDATION set...")
    evaluator_val = ModelEvaluator(best_model_path, config.output_path)
    metrics_val = evaluator_val.evaluate('val')  # FIXED: Changed from 'valid' to 'val'
else:
    print(f"⚠️ Model not found: {best_model_path}")

In [ ]:
# ============================================================================
# CELL 13: Evaluate on Test Set
# ============================================================================

if Path(best_model_path).exists():
    print("📊 Evaluating on TEST set...")
    evaluator_test = ModelEvaluator(best_model_path, config.output_path)
    metrics_test = evaluator_test.evaluate('test')
else:
    print(f"⚠️ Model not found: {best_model_path}")

In [ ]:
# ============================================================================
# CELL 14: Export Model to ONNX (Optional)
# ============================================================================

if Path(best_model_path).exists():
    print("📦 Exporting model to ONNX format...")
    best_model = YOLO(best_model_path)

    try:
        best_model.export(
            format='onnx',
            imgsz=config.image_size,
            dynamic=False,
            simplify=True
        )
        print(f"✅ ONNX export successful!")
        print(f"📁 {config.runs_path}/{config.experiment_name}/weights/best.onnx")
    except Exception as e:
        print(f"⚠️ ONNX export failed: {e}")
else:
    print(f"⚠️ Model not found: {best_model_path}")

In [ ]:
# ============================================================================
# CELL 15: Download Results (Optional - for Colab)
# ============================================================================

try:
    from google.colab import files

    print("📥 Downloading model files...")

    # Download best model
    if Path(f'{config.runs_path}/{config.experiment_name}/weights/best.pt').exists():
        files.download(f'{config.runs_path}/{config.experiment_name}/weights/best.pt')

    # Download ONNX if available
    onnx_path = f'{config.runs_path}/{config.experiment_name}/weights/best.onnx'
    if Path(onnx_path).exists():
        files.download(onnx_path)

    # Download visualizations
    print("\n📥 Downloading visualizations...")
    for viz_file in [
        'confusion_matrix_val.png',
        'confusion_matrix_test.png',
        'per_class_metrics_test.png',
        'confidence_distribution_test.png',
        'confidence_vs_accuracy_test.png',
        'augmentation_preview.png'
    ]:
        viz_path = f'/content/{viz_file}'
        if Path(viz_path).exists():
            files.download(viz_path)

    print("\n✅ All files downloaded!")

except ImportError:
    print("⚠️ Not running in Colab, skipping download.")
except Exception as e:
    print(f"⚠️ Download failed: {e}")